# Facet benchmark: how accurate is each reconstruction method?

This notebook measures how close Facet's 3D models come to the real object. It takes
objects from **Google Scanned Objects** (real household items scanned in 3D by Google Research),
renders one photo of each, runs the pipeline on that photo, and compares the result with the
scan, for both reconstruction methods:

* **InstantMesh** (the default): a feed-forward network, one pass, seconds;
* **NeuS** (the "classic" mode): fits a neural surface to the views, minutes.

It reports Chamfer distance, F-score, normal consistency, silhouette IoU and PSNR, plus time and
GPU memory, as a table and a chart. The method is explained in
[`docs/benchmark.md`](https://github.com/Kareemwafaa/Image-to-3D-mesh/blob/main/docs/benchmark.md).

**Setup (Kaggle):** right sidebar → *Session options* → *Accelerator* → **GPU T4 x2**, and
*Internet* → **On**.

**How long:** about 5 to 6 hours for the default 30 objects with both methods and the
ground-truth-views run (NeuS takes about 5 minutes per fit). Kaggle stops an interactive
session when you close the tab, so use **Save Version → Save & Run All (Commit)**: it runs in the
background for up to 12 hours and keeps everything written to `/kaggle/working`. For a quick
try, set `N_OBJECTS = 3` first.

In [ ]:
# Settings
REPO_URL = "https://github.com/Kareemwafaa/Image-to-3D-mesh"
BRANCH = "main"
N_OBJECTS = 30                 # size of the fixed Google Scanned Objects subset
METHODS = "instantmesh,neus"   # comma-separated
ORACLE = True                  # also reconstruct from perfect (ground-truth) views
NEUS_ITERS = 2500              # same as the app's classic mode
TIME_LIMIT_MIN = 660           # stop starting new objects after 11 h (Kaggle's limit is 12 h)
OUT_DIR = "/kaggle/working/bench" if __import__("os").path.isdir("/kaggle") else "/content/bench"

## 1. Check the GPU

In [ ]:
import shutil, subprocess

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU found. Kaggle: Session options > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

## 2. Get the code and install it

The same pinned environment the app uses (`/tmp/facet-env`), 1 to 3 minutes.

In [ ]:
import os

REPO_DIR = "/kaggle/working/facet" if os.path.isdir("/kaggle") else "/content/facet"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
!bash {REPO_DIR}/scripts/setup_gpu_env.sh

## 3. Run the benchmark

Downloads the objects (a few MB each) and the models (about 10 GB, the first time), then works
through the objects one by one and prints a line per step. Results are saved after every object,
so if the run stops you can run this cell again and it continues where it left off.

In [ ]:
oracle_flag = "--oracle" if ORACLE else ""
!cd {REPO_DIR}/backend && PYTHONUNBUFFERED=1 /tmp/facet-env/bin/python -m app.bench run \
    --out {OUT_DIR} --objects {N_OBJECTS} --methods {METHODS} {oracle_flag} \
    --recon-iters {NEUS_ITERS} --time-limit {TIME_LIMIT_MIN}

## 4. Results

In [ ]:
from IPython.display import Markdown, SVG, display

!cd {REPO_DIR}/backend && /tmp/facet-env/bin/python -m app.bench report --out {OUT_DIR} > /dev/null
display(Markdown(open(f"{OUT_DIR}/results.md").read()))
display(SVG(filename=f"{OUT_DIR}/chart.svg"))

## 5. Download

Everything is in `bench/` under *Output* (right sidebar, or the version's *Output* tab):

* `results.md`, `chart.svg`: the table and chart for the README;
* `results.csv`, `results.jsonl`: every number, per object;
* `subset.json`: which objects were used;
* `objects/<name>/`: the input photo, the six views and each method's aligned mesh (`.ply`).

To put the numbers in the repo, send `results.md`, `chart.svg`, `results.csv` and `subset.json`.

In [ ]:
!cd {OUT_DIR}/.. && zip -qr bench.zip bench -x "bench/objects/*" && ls -lh bench.zip